# Curs 2 — Ecosistemul de modele

Scopul acestui notebook: testăm **2-3 modele diferite** pe același input și alegem modelul potrivit pentru proiect.

Vom folosi:
1. **Gemini** — providerul principal, prin cheia obținută din Google AI Studio.
2. **OpenRouter** — provider alternativ, util pentru comparație și backup când Gemini are limite de quota.
## OpenRouter — de unde luăm cheia
1. Intră pe https://openrouter.ai/
2. Creează cont sau autentifică-te.
3. Mergi la **Keys**.
4. Creează un nou API key.
5. Copiază cheia în fișierul `.env`:
```env
OPENROUTER_API_KEY=pune-cheia-ta-aici
---

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

## 1. Configurare — mai multe modele

In [2]:
MODELE = [
    ("gemini", "gemini-2.5-flash", "Gemini 2.5 Flash"),
    ("gemini", "gemini-2.5-flash-lite", "Gemini 2.5 Flash Lite"),
    ("openrouter", "openrouter/free", "OpenRouter Free"),
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['Gemini 2.5 Flash', 'Gemini 2.5 Flash Lite', 'OpenRouter Free']


In [3]:
# Configurăm providerii și cheile API din fișierul .env

load_dotenv()

BASE_URLS = {
    "gemini": "https://generativelanguage.googleapis.com/v1beta/openai/",
    "openrouter": "https://openrouter.ai/api/v1"
}

API_KEYS = {
    "gemini": os.getenv("GEMINI_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY")
}

def make_client(provider):
    """Creează clientul API pentru providerul ales."""
    return OpenAI(
        api_key=API_KEYS[provider],
        base_url=BASE_URLS[provider]
    )

## 2. Funcție helper — trimitem același prompt la orice model

În loc să scriem același cod de 3 ori, facem o funcție.

In [5]:
# varianta minimala + retry pentru erori temporare (503)

import time

from openai import InternalServerError



client = make_client("gemini")

prompt = "Explică în 2 propoziții ce este un Labubu."



def call_with_retry(client, model, prompt, retries=3, base_delay=1.5):

    for attempt in range(retries):

        try:

            response = client.chat.completions.create(

                model=model,

                messages=[{"role": "user", "content": prompt}]

            )

            return response.choices[0].message.content

        except InternalServerError as e:

            # 503 = suprasarcina temporara la provider; asteptam si reincercam

            if attempt == retries - 1:

                raise

            time.sleep(base_delay * (2 ** attempt))



print(call_with_retry(client, "gemini-2.5-flash-lite", prompt))



# cu functie

def ask(provider, model, prompt):

    client = make_client(provider)

    return call_with_retry(client, model, prompt)



# iar functia poate fi apelata astfel:

raspuns = ask(

    provider="gemini",

    model="gemini-2.5-flash-lite",

    prompt="Explică în 2 propoziții ce este un LLM."

)



print(raspuns)


Labubu este o creatură fantastică, un personaj de pluș și o figurină de colecție, cunoscută pentru aspectul său drăguț, dar ușor demonic. Este popular în cultura pop asiatică și internațională datorită designului său unic și expresiilor sale simpatice.
Un LLM (Large Language Model) este un tip de inteligență artificială antrenat pe cantități masive de text pentru a înțelege, genera și manipula limbajul uman. Aceste modele pot realiza sarcini variate, de la răspunsuri la întrebări și rezumarea textelor, până la traduceri și scrierea de conținut creativ.


In [6]:
from openai import RateLimitError, APIError, AuthenticationError, InternalServerError

import json

import time



def ask(provider, model, prompt, system=None, temperature=0.7, json_schema=None, retries=3, fallback_model=None):

    """Trimite un prompt la model. Retry pentru erori temporare si fallback optional."""



    client = make_client(provider)



    messages = []



    if system:

        messages.append({"role": "system", "content": system})



    messages.append({"role": "user", "content": prompt})

    extra_args = {}



    if json_schema:

        extra_args["response_format"] = {

            "type": "json_schema",

            "json_schema": json_schema

        }



    models_to_try = [model]

    if fallback_model and fallback_model != model:

        models_to_try.append(fallback_model)



    for model_name in models_to_try:

        for attempt in range(retries):

            try:

                response = client.chat.completions.create(

                    model=model_name,

                    messages=messages,

                    temperature=temperature,

                    **extra_args

                )



                text = response.choices[0].message.content.strip()



                if json_schema:

                    return json.loads(text)



                return text



            except (RateLimitError, InternalServerError):

                if attempt == retries - 1:

                    break

                time.sleep(1.5 * (2 ** attempt))



            except AuthenticationError:

                return "[Eroare: API key invalidă sau lipsă. Verifică .env.]"



            except APIError as e:

                return f"[Eroare API: {e}]"



            except Exception as e:

                return f"[Eroare: {type(e).__name__} — {e}]"



    return f"[Eroare: modelul {model} este indisponibil momentan. Încearcă din nou peste 1-2 minute sau folosește fallback.]"


## 3. Test 1 — Calitatea pe limba română

Testăm dacă modelele înțeleg și răspund corect în română.

In [7]:
PROMPT_RO = """
Ești un scriitor care adoptă un ton conspiraționist și suspicios. Rezumă în exact 2 propoziții scurte, în română, principalele schimbări din politica românească din ultimii 5 ani, sugerând motive ascunse, coincidențe și legături posibile (ton conspirativ). Nu inventa fapte: exprimă doar speculații clar marcate folosind formulări precum "ar putea", "se pare", "unele surse sugerează". Maxim 80 de cuvinte. 
"""


for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    raspuns = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT_RO,
        temperature=0.2
    )

    print(raspuns)


--- Gemini 2.5 Flash ---
Ultimii cinci ani au reconfigurat șocant scena politică, foștii adversari PSD și PNL unindu-se într-o coaliție nefirească, **ceea ce ar putea sugera** o înțelegere secretă pentru controlul total al puterii. **Se pare că** această alianță, cu rotația premierilor, a fost orchestrată pentru a menține un status quo, în timp ce **unele surse sugerează** ascensiunea unor partide extremiste ca o diversiune calculată, distrăgând de la adevăratele jocuri de culise.

--- Gemini 2.5 Flash Lite ---
În ultimii cinci ani, s-au observat manevre politice subtile, menite să consolideze anumite interese, iar coincidențele din spatele unor decizii importante ar putea ascunde o agendă mai amplă. Se pare că anumite schimbări legislative au fost orchestrate pentru a favoriza grupuri specifice, iar unele surse sugerează că totul face parte dintr-un plan bine pus la punct.

--- OpenRouter Free ---
Se pare căschimbările politice recente ar putea fi mai mult decât simple ajustări, cu u

## 4. Test 2 — Urmează instrucțiunile din system prompt+ adnotare

Vedem dacă modelele respectă rolul dat prin `system`.

In [8]:
SYSTEM = """
Ești un asistent de cercetare care adnotează comentarii politice într-un ton conspiraționist și suspicios.
Adoptă exprimări speculativ-conspirative folosind formulări ca "ar putea", "se pare", "unele surse sugerează".
Răspunzi scurt, clar și nu inventezi informații verificate.
"""

PROMPT = """
Analizează următorul comentariu politic:
"Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."

Răspunde în 4 linii:
Ton:
Emoție dominantă:
Țintă principală:
Populism: da/nu
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0
    ))


--- Gemini 2.5 Flash ---
[Eroare: modelul gemini-2.5-flash este indisponibil momentan. Încearcă din nou peste 1-2 minute sau folosește fallback.]

--- Gemini 2.5 Flash Lite ---
Ton: Conspiraționist, suspicios.
Emoție dominantă: Furie, neîncredere.
Țintă principală: Clasa politică.
Populism: Da.

--- OpenRouter Free ---
Ton: critic și deziluzionat  
Emoție dominantă: furie/indignare  
Țintă principală: politicienii și clasa politică  
Populism: da


## 5. Test 3 — Output structurat (JSON)

Agenții noștri vor trebui să returneze date structurate.
Testăm dacă modelele pot produce JSON valid la cerere.

In [9]:
SCHEMA_ADNOTARE = {
    "name": "adnotare_comentariu_politic",
    "schema": {
        "type": "object",
        "properties": {
            "ton": {
                "type": "string",
                "enum": ["pozitiv", "negativ", "neutru"]
            },
            "emotie_dominanta": {
                "type": "string",
                "enum": ["furie", "frica", "speranta", "dezamagire", "ironie", "neutru"]
            },
            "tinta_principala": {
                "type": "string"
            },
            "populism": {
                "type": "boolean"
            },
            "explicatie_scurta": {
                "type": "string"
            }
        },
        "required": [
            "ton",
            "emotie_dominanta",
            "tinta_principala",
            "populism",
            "explicatie_scurta"
        ],
        "additionalProperties": False
    }
}

In [10]:
COMENTARIU = "Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."

SYSTEM = "Ești un asistent de cercetare care adnotează comentarii politice."

PROMPT = f"Adnotează următorul comentariu politic: {COMENTARIU}"

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    rezultat = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.1,
        json_schema=SCHEMA_ADNOTARE
    )

    print(rezultat)


--- Gemini 2.5 Flash ---
{'ton': 'negativ', 'emotie_dominanta': 'furie', 'tinta_principala': 'politicienii', 'populism': True, 'explicatie_scurta': 'Comentariul exprimă furie și dezamăgire față de clasa politică, acuzând-o de corupție și de ignorarea nevoilor oamenilor simpli, folosind o retorică populistă.'}

--- Gemini 2.5 Flash Lite ---
{'ton': 'negativ', 'emotie_dominanta': 'furie', 'tinta_principala': 'politicieni', 'populism': True, 'explicatie_scurta': 'Comentariul exprimă o furie generalizată față de politicieni, acuzându-i de corupție și ignorarea voinței poporului, un discurs tipic populist.'}

--- OpenRouter Free ---
{'emotie_dominanta': 'furie', 'explicatie_scurta': 'Deu, oamenii plătesc nota nu sunt mai delete, oameni sunt platonici, mai delete. Așa da, urma politice este o toate parte, iar oameni, delete, sunt oameni care nu sunt delete. Nenestruite, oameni sunt oameni care nu sunt delete, iar oameni sunt oameni care nu sunt delete. Așa da, urma politice este o toate par

## 6. Test 4 — Stabilitate la temperature diferite

Un model bun pentru agenți trebuie să fie **stabil** — același input, răspunsuri similare.
Testăm cu Gemini (poți schimba cu orice model).

In [11]:
PROMPT_STAB = """
Curtea Constituțională a anulat alegerile.
Explică în exact 2 propoziții ce ar putea sugera acest lucru despre culisele puterii, jocurile de influență și posibilele interese ascunse din viața politică.
Răspunde într-un ton conspiraționist și suspicios, dar nu prezenta speculațiile ca fapte; marchează-le clar ca interpretări neconfirmate.
"""

TEMPERATURI = [0.1, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ Gemini 2.5 Flash ]

temperature=0.1:
Anularea alegerilor de către Curtea Constituțională ar putea sugera că decizia nu a fost una pur juridică, ci mai degrabă o mișcare calculată într-un joc de putere mult mai amplu, dictată de interese oculte. Aceasta ar putea indica o intervenție a unor forțe nevăzute, care trag sforile din culise pentru a reconfigura scena politică, posibil pentru a bloca ascensiunea unor actori incomozi sau pentru a proteja anumite privilegii și agende ascunse.

temperature=0.7:
Anularea alegerilor de către Curtea Constituțională ar putea sugera o intervenție masivă și coordonată din culise, unde interese obscure ar fi manipulat pârghiile justiției pentru a anula un rezultat considerat indezirabil. Această mișcare ar putea fi interpretată ca o resetare forțată a scenei politice, orchestrată de centre de putere invizibile care își impun agenda, indiferent de voința populară exprimată la urne.

tempera

## 7. Alegerea modelului pentru proiect

Completați tabelul după testele de mai sus. Nu căutați „cel mai bun model” în general, ci modelul cel mai potrivit pentru proiectul vostru.
| Model | Răspunde bine în română? | Respectă instrucțiunile? | Merge pentru adnotare? | Are erori / quota? | Observație scurtă |
|---|---|---|---|---|---|
| Gemini 2.5 Flash Lite | da | da  | da  | nu | e ok, modest, dar mai bun este gemini 2.5 flash |
| OpenRouter Free | parțial | da | parțial | nu | are greseli gramaticale |
| Gemini 2.5 flash| da / nu / parțial | da / nu / parțial | da / nu / parțial | da / nu | cel mai bun comparativ cu celelalte |
### Decizie
**Model principal ales: Gemini 2.5 flash**  
**Model de rezervă: Gemini 2.5 Flash Lite**  
**Temperature recomandată: 0.7**  
**De ce am ales acest model? A dovedit ca este cel mai coerent și stabil model. Are cea mai buna calitate a răpsunsurilor oferite dpdv. al limbii române. Poate fi folosit pentru adnotarea comentariilor.**  


## 8. Configurația finală a proiectului

putem să copiem asta in core/config.py

In [ ]:
# core/config.py
# Configurația modelului ales de echipă după testele din Cursul 2.
# Nu puneți chei API aici. Cheile rămân doar în fișierul local .env.
PROVIDER_PRINCIPAL = "gemini"
MODEL_PRINCIPAL = "gemini-2.5-flash"
PROVIDER_FALLBACK = "gemini"
MODEL_FALLBACK = "gemini-2.5-flash-lite"
TEMPERATURE = 0.7

---

## Livrabile C2

Până la cursul următor:

- [ ] Notebook completat cu 2-3 modele testate
- [ ] Matricea de decizie completată cu observații reale
- [ ] README actualizat cu modelul ales și justificarea
- [ ] `.env` configurat cu cheia pentru modelul ales